In [6]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')

In [10]:
from pathlib import Path
import math
import time
import random
import datetime
from functools import partial

import torch
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2

from computer_vision.torch_video.utils.plotting import plot_all
from computer_vision.video_mae.parameter_parser import parser
from computer_vision.video_mae.dataset.pretrained_datasets import HybridVideoMAE, DataAugmentationForVideoMAEv2
from computer_vision.video_mae.models.modeling_pretrain import PretrainVisionTransformer, pretrain_videomae_tiny_patch16_224
from computer_vision.video_mae.utils import seed_worker, multiple_pretrain_samples_collate, cosine_scheduler, get_grad_norm
from computer_vision.video_mae.optim_factory import create_optimizer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'

mini_train=False
if not mini_train:
    output_dirpath=Path('D:/results/ucf101/video_mae/train') 
    arguments= f"""--data_root {root} --data_path {annotation_path} --output_dir {output_dirpath} 
    --mask_type tube --mask_ratio 0.9 --decoder_mask_type run_cell --decoder_mask_ratio 0.5
    --decoder_depth 4 --with_checkpoint --cos_attn --num_frame 16 --sampling_rate 4 --num_sample 1 
    --opt adamw --lr 6e-4 --clip_grad 0.02 --opt_betas 0.9 0.95 --warmup_epochs 30 
    --batch-size 24 --num_workers 0 --print_freq 30 --time 13
    --device cuda --epochs 300 --resume
    """ # --use-cutmix-mixup
else:
    output_dirpath=Path('D:/results/ucf101/video_mae/mini_train') 
    arguments= f"""--data_root {root} --data_path {annotation_path}  --output_dir {output_dirpath}  
    --mask_type tube --mask_ratio 0.9 --decoder_mask_type run_cell --decoder_mask_ratio 0.5
    --decoder_depth 4 --with_checkpoint --cos_attn --num_frame 16 --sampling_rate 4 --num_sample 1 
    --opt adamw --lr 6e-4 --clip_grad 0.02 --opt_betas 0.9 0.95 --warmup_epochs 30 
    --batch-size 24 --num_workers 0 --print_freq 20  --plot_freq 3
    --epochs 300 --device cuda --n_steps 60  --n_epochs 6 --time 0.5 --resume
    """ # --use-cutmix-mixup --time 18 --resume
args=parser.parse_args(arguments.split())


args.output_dir=Path(args.output_dir)
args.output_dir.mkdir(parents=True, exist_ok=True)
args.checkpoint_dir=args.output_dir/"checkpoints"
args.checkpoint_dir.mkdir(parents=True, exist_ok=True)
args.last=args.checkpoint_dir/args.last
args.best=args.checkpoint_dir/args.best
print(f"{args.last=}")
print(f"{args.best=}")

args.last=WindowsPath('D:/results/ucf101/video_mae/train/checkpoints/last.pth')
args.best=WindowsPath('D:/results/ucf101/video_mae/train/checkpoints/best.pth')


In [11]:
plot_all(args.output_dir/"result.csv")